# 01b — Amazon Clothing Data & Embeddings

**input:** `data/amazon/meta_Clothing_Shoes_and_Jewelry.jsonl.gz`

**Does one thing:**
1. Reads only the first 10,000 items
2. Builds a text description
3. Encodes with SBERT
4. Saves to disk

**output:** `data/amazon/items.pkl` and `data/amazon/embeddings.npy`

In [ ]:
import numpy as np
import pandas as pd
import gzip, json
from pathlib import Path
from sentence_transformers import SentenceTransformer

DATA_DIR   = Path('../data/amazon')
DATA_DIR.mkdir(exist_ok=True)

SBERT_MODEL = 'all-MiniLM-L6-v2'
MAX_ITEMS   = 10_000   # only first 10k items

print('imports OK')

c:\Users\Sanaz\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


### Step 1 — Read metadata (first 10k items)

In [ ]:
META_PATH = DATA_DIR / 'meta_Clothing_Shoes_and_Jewelry.jsonl.gz'

records = []
with gzip.open(META_PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= MAX_ITEMS:
            break
        try:
            item = json.loads(line)
            records.append(item)
        except:
            continue
        if (i+1) % 2000 == 0:
            print(f'  Read {i+1} items...')

print(f'\nTotal loaded: {len(records)}')
print('\nKeys in first item:')
print(list(records[0].keys()))
print('\nFirst item sample:')
print(json.dumps(records[0], indent=2)[:500])

  Read 2000 items...
  Read 4000 items...
  Read 6000 items...
  Read 8000 items...
  Read 10000 items...

Total loaded: 10000

Keys in first item:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

First item sample:
{
  "main_category": "AMAZON FASHION",
  "title": "BALEAF Women's Long Sleeve Zip Beach Coverup UPF 50+ Sun Protection Hooded Cover Up Shirt Dress with Pockets",
  "average_rating": 4.2,
  "rating_number": 422,
  "features": [
    "90% Polyester, 10% Spandex",
    "Zipper closure",
    "Machine Wash",
    "Long sleeve sun protection coverups--UPF 50+ blocks the sun from burning",
    "Zipped v-neckline--fashionable V neck and smooth 1/4 zipper allows to staying place as you like",
    "Two drop-


### Step 2 — Build DataFrame

In [ ]:
def extract_item(item: dict) -> dict:
    """
    Extracts relevant fields from each item.
    Field names may differ across dataset versions.
    """
    # title
    title = item.get('title', item.get('name', ''))
    if isinstance(title, list):
        title = title[0] if title else ''

    # category
    cats = item.get('categories', item.get('category', []))
    if isinstance(cats, list) and cats:
        if isinstance(cats[0], list):
            cats = cats[0]  # nested list
        category = ' > '.join(str(c) for c in cats[-3:])  # last 3 category levels
    else:
        category = str(cats) if cats else 'Unknown'

    # description
    desc = item.get('description', '')
    if isinstance(desc, list):
        desc = ' '.join(desc[:2])  # only first 2 sentences
    desc = str(desc)[:200]  # truncate to 200 characters

    # price
    price = item.get('price', '')

    # asin (product id)
    asin = item.get('asin', item.get('parent_asin', str(len(records))))

    return {
        'asin':     asin,
        'title':    str(title)[:150],
        'category': category,
        'description': desc,
        'price':    price,
    }


items_raw = [extract_item(r) for r in records]
items = pd.DataFrame(items_raw)

# remove items with no title
items = items[items['title'].str.len() > 3].reset_index(drop=True)

# text for embedding: title + category
items['text'] = items['title'] + ' | ' + items['category']

print(f'Items after cleaning: {len(items)}')
print()
items[['title', 'category', 'text']].head(5)

Items after cleaning: 9999



,title,category,text
0,BALEAF Women's Long Sleeve Zip Beach Coverup U...,Clothing > Swimsuits & Cover Ups > Cover-Ups,BALEAF Women's Long Sleeve Zip Beach Coverup U...
1,Merrell Work Moab 2 Vent Waterproof SR Boulder,Outdoor > Hiking & Trekking > Hiking Shoes,Merrell Work Moab 2 Vent Waterproof SR Boulder...
2,"SAS Women's, Relaxed Sandal",Shoes > Sandals > Flats,"SAS Women's, Relaxed Sandal | Shoes > Sandals ..."
3,SheIn Women's Basic Stretch Plaid Mini Bodycon...,Novelty > Women > Skirts,SheIn Women's Basic Stretch Plaid Mini Bodycon...
4,"Michael Kors Cindy, Women’s Cross-Body Bag",Women > Handbags & Wallets > Crossbody Bags,"Michael Kors Cindy, Women’s Cross-Body Bag | W..."


### Step 3 — Embed

In [ ]:
EMB_PATH = DATA_DIR / 'embeddings.npy'

if EMB_PATH.exists():
    print('Loading cached embeddings...')
    embeddings = np.load(EMB_PATH)
else:
    print(f'Encoding {len(items)} items...')
    model = SentenceTransformer(SBERT_MODEL)
    embeddings = model.encode(
        items['text'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(EMB_PATH, embeddings)
    print('Saved to cache.')

print(f'Shape: {embeddings.shape}')

Encoding 9999 items...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 370.30it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 40/40 [02:06<00:00,  3.17s/it]


Saved to cache.
Shape: (9999, 384)


### Step 4 — Save & Preview

In [ ]:
items.to_pickle(DATA_DIR / 'items.pkl')

print('Saved: data/amazon/items.pkl')
print('Saved: data/amazon/embeddings.npy')
print()
print('Category distribution (top 10):')
print(items['category'].value_counts().head(10).to_string())
print()
print('✅ notebook 01b complete')

Saved: data/amazon/items.pkl
Saved: data/amazon/embeddings.npy

Category distribution (top 10):
category
Clothing > Dresses > Casual                                       169
Clothing > Tops, Tees & Blouses > Blouses & Button-Down Shirts    164
Athletic > Running > Road Running                                 163
Men > Shirts > T-Shirts                                           148
Women > Clothing > Dresses                                        148
Clothing > Tops, Tees & Blouses > T-Shirts                        145
Women > Shoes > Fashion Sneakers                                  141
Clothing > Tops, Tees & Blouses > Tunics                          131
Women > Shoes > Pumps                                             121
Lingerie > Bras > Everyday Bras                                   113

✅ notebook 01b complete
